# Feature Exploration

In this notebook, we validate features by looking at:
- Missing rate checks
- Distribution plots
- Correlation vs target
- Feature importance preview

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

## 1. Load Data
Load the dataset for exploration. Adjust the path and target column as needed.

In [ ]:
# df = pd.read_csv('../data/raw/loan_data.csv')
# target_col = 'default_status'

# For demonstration, creating a dummy dataframe
np.random.seed(42)
n_samples = 1000
df = pd.DataFrame({
    'loan_amount': np.random.normal(10000, 5000, n_samples),
    'interest_rate': np.random.uniform(5, 25, n_samples),
    'credit_score': np.random.normal(650, 100, n_samples),
    'income': np.random.normal(60000, 20000, n_samples),
    'age': np.random.randint(20, 70, n_samples),
    'default_status': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2])
})

# Introduce some missing values
df.loc[np.random.choice(df.index, 50), 'income'] = np.nan
df.loc[np.random.choice(df.index, 20), 'credit_score'] = np.nan

target_col = 'default_status'
df.head()

## 2. Missing Rate Check

In [ ]:
missing_rates = df.isnull().mean() * 100
missing_rates = missing_rates[missing_rates > 0].sort_values(ascending=False)

plt.figure(figsize=(10, 5))
if not missing_rates.empty:
    sns.barplot(x=missing_rates.values, y=missing_rates.index)
    plt.title('Missing Value Rates (%)')
    plt.xlabel('% Missing')
    plt.ylabel('Features')
    plt.show()
else:
    print("No missing values found.")

## 3. Distribution Plots
Visualizing the distribution of numerical features.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop(target_col, errors='ignore')

n_cols = 2
n_rows = int(np.ceil(len(numeric_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(data=df, x=col, hue=target_col, kde=True, ax=axes[i], element='step')
    axes[i].set_title(f'Distribution of {col}')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

## 4. Correlation vs Target

In [ ]:
plt.figure(figsize=(8, 6))
correlations = df.corr()[target_col].drop(target_col).sort_values()
sns.barplot(x=correlations.values, y=correlations.index)
plt.title(f'Feature Correlation with {target_col}')
plt.xlabel('Correlation Coefficient')
plt.show()

## 5. Feature Importance Preview
Using a simple Random Forest model to gauge initial feature importance.

In [ ]:
# Prepare data for RF (handle missing values simply for preview)
df_rf = df.copy()
for col in df_rf.columns:
    if df_rf[col].dtype in [np.float64, np.int64]:
        df_rf[col] = df_rf[col].fillna(df_rf[col].median())
    else:
        df_rf[col] = df_rf[col].fillna(df_rf[col].mode()[0])
        
X = df_rf.drop(columns=[target_col])
y = df_rf[target_col]

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
sns.barplot(x=importances.values, y=importances.index)
plt.title('Random Forest Feature Importance Preview')
plt.xlabel('Importance')
plt.show()